# Synthetic Data Generation for Tool-Calling Fine-Tuning

Fine-tune a small model (`gpt-4.1-mini`) to call a catalog of tools correctly, using **training data generated synthetically by the Foundry Data Generation API** from a static OpenAPI tool spec — no production agent traffic required.

End-to-end:
1. Convert an OpenAI tool catalog to OpenAPI 3.0 and upload it
2. Generate Q&A pairs via the `ToolUseFineTuning` recipe
3. Split into train / val / test
4. Score the base model on the held-out test set with **structural tool-call matching**
5. Submit one fine-tuning job (winning hyperparameters)
6. Monitor training to completion
7. Deploy the fine-tuned model
8. Score it on the same test set and report the lift

**Result on the included Zava tool catalog**:
- Base `gpt-4.1-mini`: 9.20/10, 100% pass rate
- Fine-tuned `gpt-4.1-mini` (3 epochs, lr=1.0): **10.00/10, +8.7% lift**

**Cost**: ~$2–5 per run. **Time**: ~25–45 minutes.

This notebook is **fully self-contained** — no external skill scripts required. All helpers (OpenAI→OpenAPI conversion, datagen submission, tool-call evaluator) are defined inline.

## 1. Setup & Configuration

In [ ]:
import os, json, time, random, re, subprocess, sys
from pathlib import Path
from openai import OpenAI

# Required env vars
PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]   # https://<resource>.services.ai.azure.com/api/projects/<project>
BASE_URL         = os.environ["OPENAI_BASE_URL"]              # https://<resource>.openai.azure.com/openai/v1
API_KEY          = os.environ["AZURE_OPENAI_API_KEY"]

# Models
TEACHER_MODEL    = "gpt-4.1"        # Foundry datagen uses this to write Q&A pairs
STUDENT_MODEL    = "gpt-4.1-mini"   # the model we are fine-tuning

WORK             = Path("./run").resolve()
WORK.mkdir(exist_ok=True)

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"Project:   {PROJECT_ENDPOINT}")
print(f"Student:   {STUDENT_MODEL}")
print(f"Work dir:  {WORK}")


## 2. Inline helpers: OpenAI tools → OpenAPI 3.0 + Foundry datagen submission

These functions replace having to depend on external skill scripts. They handle the Foundry datagen API specifics (auth, polling) and the OpenAI tools-array → OpenAPI conversion the `ToolUseFineTuning` recipe expects.

In [ ]:
def openai_tools_to_openapi(openai_tools, title="Tools"):
    """Convert an OpenAI chat-completions tools array to an OpenAPI 3.0 spec.

    The Foundry ToolUseFineTuning recipe consumes OpenAPI 3.0, not the
    OpenAI tools-array. Each tool function becomes a POST operation under
    /<tool_name>.
    """
    paths = {}
    for t in openai_tools:
        fn = t.get("function", {})
        name = fn["name"]
        params = fn.get("parameters") or {"type": "object", "properties": {}}
        paths[f"/{name}"] = {
            "post": {
                "operationId": name,
                "summary": fn.get("description", ""),
                "requestBody": {
                    "required": True,
                    "content": {"application/json": {"schema": params}},
                },
                "responses": {"200": {"description": "Success"}},
            }
        }
    return {
        "openapi": "3.0.3",
        "info": {"title": title, "version": "1.0.0"},
        "paths": paths,
    }


def submit_foundry_datagen(*, file_id, recipe, teacher, max_samples,
                           output_name, project_endpoint, timeout_s=1800):
    """Submit a Foundry Data Generation API job and download the JSONL output.

    Uses the azure-ai-projects SDK. Returns the path to the downloaded JSONL.
    """
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import (
        DataGenerationJob, DataGenerationJobInputs,
        DataGenerationJobScenario, DataGenerationJobOutputOptions,
        SimpleQnADataGenerationJobOptions, ToolUseFineTuningDataGenerationJobOptions,
        FileDataGenerationJobSource, DataGenerationModelOptions,
    )
    from azure.identity import DefaultAzureCredential

    project = AIProjectClient(endpoint=project_endpoint, credential=DefaultAzureCredential())

    model_opts = DataGenerationModelOptions(model=teacher) if teacher else None
    if recipe == "tool-use":
        options = ToolUseFineTuningDataGenerationJobOptions(max_samples=max_samples, model_options=model_opts)
    else:
        options = SimpleQnADataGenerationJobOptions(max_samples=max_samples, model_options=model_opts)

    job = DataGenerationJob(inputs=DataGenerationJobInputs(
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        sources=[FileDataGenerationJobSource(file_id=file_id)],
        options=options,
        output_options=DataGenerationJobOutputOptions(name=output_name),
    ))
    print(f"Submitting datagen job (recipe={recipe}, max_samples={max_samples})...")
    created = project.datasets.create_generation_job(body=job)
    print(f"  job.id={created.id}  status={created.status}")

    deadline = time.time() + timeout_s
    while time.time() < deadline:
        j = project.datasets.get_generation_job(name=created.id)
        if str(j.status).endswith("SUCCEEDED"):
            print(f"  generated {len(j.outputs[0].file_ids) if j.outputs else 0} file(s)")
            break
        if str(j.status).endswith(("FAILED", "CANCELLED")):
            raise RuntimeError(f"datagen job ended in {j.status}")
        time.sleep(15)
    else:
        raise TimeoutError(f"datagen job did not complete in {timeout_s}s")

    out_path = WORK / f"{output_name}_dg.jsonl"
    out_blob = b""
    for fid in j.outputs[0].file_ids:
        out_blob += client.files.content(file_id=fid).read()
    out_path.write_bytes(out_blob)
    print(f"  saved {out_path.name}  ({out_path.stat().st_size:,} bytes)")
    return out_path


## 3. Convert + upload the Zava tool catalog

The bundled `fixtures/zava_tools_openai.json` has 6 retail-support tools in the OpenAI chat-completions format. We convert to OpenAPI 3.0 and upload as user_data.

In [ ]:
OPENAI_TOOLS = json.loads(Path("fixtures/zava_tools_openai.json").read_text())
print(f"Loaded {len(OPENAI_TOOLS)} tools: {[t['function']['name'] for t in OPENAI_TOOLS]}")

openapi_spec = openai_tools_to_openapi(OPENAI_TOOLS, title="Zava Resolution Desk Tools")
OPENAPI_OUT = WORK / "zava_tools_openapi.json"
OPENAPI_OUT.write_text(json.dumps(openapi_spec, indent=2))

with open(OPENAPI_OUT, "rb") as fh:
    spec_file = client.files.create(file=("zava_tools_openapi.json", fh), purpose="user_data")

for _ in range(20):
    spec_file = client.files.retrieve(file_id=spec_file.id)
    if spec_file.status == "processed": break
    time.sleep(2)
print(f"Uploaded: {spec_file.id} (status={spec_file.status})")


## 4. Generate synthetic training data

The Foundry datagen service uses the teacher (`gpt-4.1`) to write realistic user prompts plus the correct tool invocations for each. ~50 examples takes 2–5 minutes.

In [ ]:
DATAFILE = submit_foundry_datagen(
    file_id=spec_file.id,
    recipe="tool-use",
    teacher=TEACHER_MODEL,
    max_samples=60,
    output_name="zava-tools-sft",
    project_endpoint=PROJECT_ENDPOINT,
)

with open(DATAFILE, encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]
print(f"\nRows: {len(data)}")
print("First row preview:")
print(json.dumps(data[0]["messages"][:2], indent=2)[:500])


## 5. Split into train / val / test

80% train, 10% val, 10% test. The test split is held out from training so we can measure lift fairly.

In [ ]:
rng = random.Random(42)
indices = list(range(len(data))); rng.shuffle(indices)
n_train = int(0.8 * len(indices)); n_val = int(0.1 * len(indices))
train_idx = indices[:n_train]; val_idx = indices[n_train:n_train+n_val]; test_idx = indices[n_train+n_val:]

TRAIN_PATH = WORK / "train.jsonl"
VAL_PATH   = WORK / "val.jsonl"
TEST_PATH  = WORK / "test.jsonl"
for path, idxs in [(TRAIN_PATH, train_idx), (VAL_PATH, val_idx), (TEST_PATH, test_idx)]:
    with path.open("w", encoding="utf-8") as f:
        for i in idxs: f.write(json.dumps(data[i]) + "\n")

print(f"  train: {len(train_idx)} rows")
print(f"  val:   {len(val_idx)} rows")
print(f"  test:  {len(test_idx)} rows")


## 6. Baseline evaluation via the Foundry evals SDK

We use `azure-ai-evaluation.evaluate()` as the driver. Built-in evaluators don't cover tool-call structural matching, so we provide a small custom evaluator function that the SDK runs across all test rows.

In [ ]:
# pip install azure-ai-evaluation  if not already present
def tool_call_score(ref_calls, out_calls):
    """Structural match. 10 = exact match, 8 = same names different args,
    2-8 = partial name overlap, 1 = no overlap."""
    if not ref_calls: return 10 if not out_calls else 5
    if not out_calls: return 1
    def _name(c):
        if isinstance(c, dict): return ((c.get("function") or {}).get("name")) or c.get("name")
        fn = getattr(c, "function", None)
        return getattr(fn, "name", None) if fn else getattr(c, "name", None)
    def _args(c):
        if isinstance(c, dict):
            raw = ((c.get("function") or {}).get("arguments")) or c.get("arguments")
        else:
            fn = getattr(c, "function", None)
            raw = getattr(fn, "arguments", None) if fn else getattr(c, "arguments", None)
        if isinstance(raw, str):
            try: return json.loads(raw)
            except Exception: return {"_raw": raw}
        return raw or {}
    ref_names = [_name(c) for c in ref_calls]
    out_names = [_name(c) for c in out_calls]
    ref_set, out_set = set(ref_names), set(out_names)
    overlap = ref_set & out_set
    if not overlap: return 1
    if ref_set != out_set:
        ratio = len(overlap) / len(ref_set | out_set)
        return max(2, int(round(2 + ratio * 6)))
    ref_args = {_name(c): _args(c) for c in ref_calls}
    out_args = {_name(c): _args(c) for c in out_calls}
    return 10 if all(ref_args[n] == out_args.get(n) for n in ref_names) else 8


def tool_call_evaluator(*, response, ground_truth, **kwargs):
    """azure-ai-evaluation-compatible evaluator. Called per-row with the
    target's output (response) and the reference (ground_truth)."""
    ref = json.loads(ground_truth) if isinstance(ground_truth, str) else (ground_truth or [])
    out = json.loads(response) if isinstance(response, str) else (response or [])
    score = tool_call_score(ref, out)
    return {"tool_match_score": score, "tool_match_pass": 1.0 if score >= 8 else 0.0}


def make_target(model_name):
    """Returns a callable the evaluate() SDK uses to query a model row-by-row."""
    def _target(*, prompt, system, **kwargs):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[{"role":"system","content":system}, {"role":"user","content":prompt}],
                tools=OPENAI_TOOLS, temperature=0.0, max_completion_tokens=2048,
            )
            tool_calls = resp.choices[0].message.tool_calls or []
            return {"response": json.dumps([
                {"function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in tool_calls
            ])}
        except Exception as e:
            return {"response": "[]", "error": str(e)[:200]}
    return _target


# Build the data file the SDK expects
EVAL_DATA_PATH = WORK / "eval_data.jsonl"
with open(TEST_PATH, encoding="utf-8") as fin, open(EVAL_DATA_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip(): continue
        row = json.loads(line)
        msgs = row["messages"]
        system_msg = next((m.get("content") or "" for m in msgs if m.get("role") == "system"), "")
        user_msg   = next((m.get("content") or "" for m in msgs if m.get("role") == "user"), "")
        first_asst = next((m for m in msgs if m.get("role") == "assistant"), {})
        ref_tcs = first_asst.get("tool_calls") or []
        gt = [{"function": {"name": (tc.get("function") or {}).get("name"),
                            "arguments": (tc.get("function") or {}).get("arguments")}} for tc in ref_tcs]
        fout.write(json.dumps({"system": system_msg, "prompt": user_msg, "ground_truth": json.dumps(gt)}) + "\n")
print(f"Eval data: {EVAL_DATA_PATH.name}")

from azure.ai.evaluation import evaluate
print(f"\nBaseline ({STUDENT_MODEL}) evaluation...")
baseline_results = evaluate(
    data=str(EVAL_DATA_PATH),
    target=make_target(STUDENT_MODEL),
    evaluators={"tool_match": tool_call_evaluator},
    output_path=str(WORK / "baseline_eval_results.json"),
)
baseline_combined = baseline_results["metrics"].get("tool_match.tool_match_score", 0)
baseline_pass = baseline_results["metrics"].get("tool_match.tool_match_pass", 0) * 100
print(f"  Baseline: combined={baseline_combined:.2f}/10  pass_rate={baseline_pass:.1f}%")


## 7. Submit the fine-tuning job

Winning hyperparameters from prior experiments: **`gpt-4.1-mini`, 3 epochs, learning-rate multiplier 1.0**. For your own tools, sweep a few combinations and pick the winner the same way.

In [ ]:
print("Uploading train + val files...")
with open(TRAIN_PATH, "rb") as fh:
    train_file = client.files.create(file=(TRAIN_PATH.name, fh), purpose="fine-tune")
with open(VAL_PATH, "rb") as fh:
    val_file = client.files.create(file=(VAL_PATH.name, fh), purpose="fine-tune")

for f in (train_file, val_file):
    for _ in range(30):
        f = client.files.retrieve(file_id=f.id)
        if f.status == "processed": break
        time.sleep(2)
    print(f"  {f.id} status={f.status}")

print("\nSubmitting fine-tuning job...")
ft_job = client.fine_tuning.jobs.create(
    model=STUDENT_MODEL,
    training_file=train_file.id,
    validation_file=val_file.id,
    method={"type": "supervised"},
    hyperparameters={"n_epochs": 3, "learning_rate_multiplier": 1.0},
    suffix="zava-tools-demo",
    # extra_body sets the Azure-specific trainingType field that the OpenAI SDK does not model.
    extra_body={"trainingType": "globalStandard"},
)
print(f"  Job: {ft_job.id}  status={ft_job.status}")


## 8. Monitor training

Poll every 30 seconds and print step / loss as training progresses.

In [ ]:
print(f"Monitoring job {ft_job.id}...\n")
last_seen_step = -1
while True:
    job = client.fine_tuning.jobs.retrieve(ft_job.id)
    events = list(client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_job.id, limit=10))
    for e in reversed(events):
        msg = e.message or ""
        if "Step " in msg and ":" in msg:
            try:
                step = int(msg.split("Step ")[1].split(":")[0])
                if step > last_seen_step:
                    print(f"  {time.strftime('%H:%M:%S')}  {msg[:90]}")
                    last_seen_step = step
            except Exception: pass
    if job.status in ("succeeded", "failed", "cancelled"):
        print(f"\nFinal status: {job.status}")
        if job.status != "succeeded":
            print(f"Error: {job.error.message if job.error else '(none)'}")
            raise RuntimeError(f"Fine-tuning {job.status}")
        FT_MODEL_ID = job.fine_tuned_model
        print(f"Fine-tuned model: {FT_MODEL_ID}")
        break
    time.sleep(30)


## 9. Deploy the fine-tuned model

Azure requires explicit deployment to make a fine-tuned model callable. Uses the Cognitive Services management API; deployment typically becomes inferenceable in 3–5 minutes.

In [ ]:
m = re.match(r"https://([^.]+)\.openai\.azure\.com", BASE_URL)
ACCOUNT_NAME    = m.group(1)
SUBSCRIPTION_ID = os.environ.get("AZURE_SUBSCRIPTION_ID") or input("Azure subscription id: ")
RESOURCE_GROUP  = os.environ.get("AZURE_RESOURCE_GROUP") or input("Azure resource group: ")

DEPLOY_NAME = "zava-tools-ft-demo"
deploy_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}"
    f"/deployments/{DEPLOY_NAME}?api-version=2024-10-01"
)
body = {"sku": {"name": "GlobalStandard", "capacity": 100},
        "properties": {"model": {"format": "OpenAI", "name": FT_MODEL_ID, "version": "1"}}}

import urllib.request, urllib.error
token = subprocess.check_output(["az", "account", "get-access-token", "--query", "accessToken", "-o", "tsv"]).decode().strip()
req = urllib.request.Request(deploy_url, method="PUT",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    data=json.dumps(body).encode())
try: urllib.request.urlopen(req); print(f"Deployment submitted: {DEPLOY_NAME}")
except urllib.error.HTTPError as e: print(f"Deploy returned: {e.code} {e.reason}")

print("Waiting for deployment to become inferenceable (~3-5 min)...")
for i in range(20):
    try:
        client.chat.completions.create(model=DEPLOY_NAME, messages=[{"role":"user","content":"hi"}], max_completion_tokens=5)
        print(f"\n  Ready after {i*30}s"); break
    except Exception:
        time.sleep(30); print(".", end="", flush=True)


## 10. Evaluate the fine-tuned model and compare

Same test set, same evaluator, side-by-side comparison.

In [ ]:
print(f"Fine-tuned ({STUDENT_MODEL}) evaluation...")
ft_results = evaluate(
    data=str(EVAL_DATA_PATH),
    target=make_target(DEPLOY_NAME),
    evaluators={"tool_match": tool_call_evaluator},
    output_path=str(WORK / "ft_eval_results.json"),
)
ft_combined = ft_results["metrics"].get("tool_match.tool_match_score", 0)
ft_pass = ft_results["metrics"].get("tool_match.tool_match_pass", 0) * 100

print()
print("-" * 60)
print(f"  {'Model':<35}  {'Combined':>10}  {'Pass Rate':>10}")
print(f"  {'-'*35}  {'-'*10}  {'-'*10}")
print(f"  Baseline ({STUDENT_MODEL}){' '*(35-19-len(STUDENT_MODEL))}  {baseline_combined:>10.2f}  {baseline_pass:>9.1f}%")
print(f"  Fine-tuned ({STUDENT_MODEL}){' '*(35-21-len(STUDENT_MODEL))}  {ft_combined:>10.2f}  {ft_pass:>9.1f}%")

lift = (ft_combined - baseline_combined) / baseline_combined * 100 if baseline_combined > 0 else 0
print(f"\n  Lift:  {lift:+.1f}%")
if lift >= 5:
    print(f"  Fine-tuning improved combined score by {lift:+.1f}% -- ship the FT model.")
else:
    print(f"  Lift below 5% threshold. Consider: more training data, different HPs, or a larger student model.")


## Cleanup

Free the temporary deployment and uploaded files when you're done.

```python
# Delete the FT-eval deployment to free quota
req = urllib.request.Request(deploy_url, method="DELETE", headers={"Authorization": f"Bearer {token}"})
urllib.request.urlopen(req)

# Delete uploaded files
client.files.delete(spec_file.id)
client.files.delete(train_file.id)
client.files.delete(val_file.id)
```

## Bring your own tools

Replace `fixtures/zava_tools_openai.json` with your own tool catalog in the OpenAI chat-completions format. The notebook converts to OpenAPI 3.0 automatically (cell 3).

## Dependencies

```
pip install openai>=2.0 azure-ai-projects>=2.2.0 azure-identity>=1.21 azure-ai-evaluation>=1.0
```